# Keyboard audio: automatic labels, clip ranges, FFT and spectrograms
Run locally in Windows Jupyter or VS Code. The data root is prefilled below. This version reads the recorder's `key_events.csv` automatically: `time_seconds,event,key,character,audio_sample_index`.

Each **down/up pair** identifies the key and its held interval. We add a 50 ms lead-in and at least 100 ms after release, with a minimum 300 ms after key-down. The shaded region is the actual audio analyzed. No manual labeling is needed. Synchronization is approximate: padding helps, but audio/event alignment still needs spot checks.

The recording files remain unchanged. The final optional export writes new clips and a manifest only if enabled.

In [ ]:
# Run once if dependencies are missing, then restart the kernel if requested.
%pip install numpy pandas scipy matplotlib soundfile

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy import signal
from IPython.display import display, Audio
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.grid': True})
ROOT = Path(r'C:\Users\quang\Documents\KeyboardAudioSessions')
if not ROOT.is_dir():
    raise FileNotFoundError(f'Folder not found: {ROOT}. Run locally or change ROOT.')
wavs = sorted(ROOT.rglob('*.wav'))
if not wavs:
    raise FileNotFoundError('No WAV files found below ROOT.')
rows = []
for p in wavs:
    try:
        info = sf.info(str(p))
        rows.append(dict(path=str(p.relative_to(ROOT)), seconds=info.duration,
                         sample_rate=info.samplerate, channels=info.channels, format=info.subtype))
    except Exception as exc:
        rows.append(dict(path=str(p.relative_to(ROOT)), error=str(exc)))
display(pd.DataFrame(rows).rename_axis('session_index'))

## Select a recording
Change `SESSION_INDEX` using the inventory above. Stereo channels are shown separately. Analysis defaults to channel 0, avoiding accidental cancellation from averaging channels.

In [ ]:
SESSION_INDEX = 0
CHANNEL = 0                 # 0 = first/left; 1 = second/right, if present
wav_path = wavs[SESSION_INDEX]
audio, fs = sf.read(str(wav_path), always_2d=True, dtype='float64')
if not 0 <= CHANNEL < audio.shape[1]:
    raise ValueError('CHANNEL is outside the available channel range.')
x = audio[:, CHANNEL]
duration = len(x) / fs
print('Selected:', wav_path)
print(f'{fs:,} Hz | {audio.shape[1]} channel(s) | {duration:.2f} seconds')
quality = pd.DataFrame({
    'channel': range(audio.shape[1]),
    'peak_abs': np.max(np.abs(audio), axis=0),
    'rms': np.sqrt(np.mean(audio**2, axis=0)),
    'near_full_scale_percent': 100*np.mean(np.abs(audio) >= 0.999, axis=0)
})
display(quality)
print('Near-full-scale samples are a clipping warning, not a complete clipping diagnosis.')
fig, axes = plt.subplots(audio.shape[1], 1, squeeze=False, sharex=True,
                         figsize=(12, 3*audio.shape[1]))
stride = max(1, len(x)//200000)
for c, ax in enumerate(axes[:, 0]):
    ax.plot(np.arange(0,len(x),stride)/fs, audio[::stride,c], lw=0.6)
    ax.set(ylabel=f'Channel {c}\nAmplitude (FS)', title=wav_path.parent.name if c == 0 else '')
axes[-1,0].set_xlabel('Time from WAV start (s)')
plt.tight_layout()
display(Audio(audio.T if audio.shape[1] <= 2 else x, rate=fs, normalize=False))

## Load the recorder's existing labels
CSV labels are authoritative for key identity. Audio sample indices are preferred when present; elapsed seconds are the fallback. Key-up is paired to the corresponding key-down; repeated down events while held are ignored. Missing releases and window collisions are flagged, not silently accepted for export.

In [ ]:
csv_path = wav_path.parent / 'key_events.csv'
events = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
print('CSV:', csv_path if csv_path.exists() else 'not found; use manual time selection below')
print('Columns:', events.columns.tolist())
display(events.head(15))
meta_path = wav_path.parent / 'session.json'
if meta_path.exists():
    print(meta_path.read_text(encoding='utf-8-sig')[:8000])

In [ ]:
PRE_SECONDS = 0.05
MIN_POST_DOWN_SECONDS = 0.30
POST_RELEASE_SECONDS = 0.10
MAX_HOLD_SECONDS = 1.0
AUDIO_OFFSET_SECONDS = 0.0  # default: preserve recorder timing; change only after alignment checks

def make_clip_table(events, sample_rate, n_samples):
    required = {'time_seconds', 'event', 'key'}
    if not required.issubset(events.columns):
        raise ValueError(f'CSV needs {sorted(required)}; found {events.columns.tolist()}')
    e = events.copy()
    e['source_row'] = np.arange(len(e))
    seconds = pd.to_numeric(e['time_seconds'], errors='coerce')
    e['time_s'] = seconds
    if 'audio_sample_index' in e:
        samples = pd.to_numeric(e['audio_sample_index'], errors='coerce')
        mismatch = (samples/sample_rate - seconds).abs()
        if (mismatch > 0.002).any():
            raise ValueError('Sample indices disagree with elapsed seconds by >2 ms. Check WAV/CSV pairing and rate.')
        e['time_s'] = (samples/sample_rate).where(samples.notna(), seconds)
    if e['time_s'].isna().any():
        raise ValueError('Non-numeric event times found; inspect CSV before extraction.')
    e['time_s'] += AUDIO_OFFSET_SECONDS
    e['event'] = e['event'].astype(str).str.strip().str.lower()
    e['key'] = e['key'].astype(str)
    e = e.sort_values(['time_s','source_row'], kind='stable')
    held, records = {}, []
    ignored_repeats, unmatched_ups = 0, 0
    for r in e.itertuples(index=False):
        key = r.key
        if r.event == 'down':
            if key in held:
                ignored_repeats += 1
                continue
            held[key] = (r.time_s, r.source_row)
        elif r.event == 'up':
            if key not in held:
                unmatched_ups += 1
                continue
            down, row_id = held.pop(key)
            records.append(dict(key=key,time_s=down,up_s=r.time_s,source_row=row_id))
    for key,(down,row_id) in held.items():
        records.append(dict(key=key,time_s=down,up_s=np.nan,source_row=row_id))
    cols=['key','time_s','up_s','source_row']
    table=pd.DataFrame(records,columns=cols).sort_values('time_s').reset_index(drop=True)
    if table.empty:
        raise ValueError('No key-down events found.')
    statuses=[]
    starts=[]; ends=[]
    for r in table.itertuples():
        flags=[]
        if pd.isna(r.up_s): flags.append('missing_key_up')
        if pd.notna(r.up_s) and r.up_s-r.time_s > MAX_HOLD_SECONDS:
            flags.append('long_hold')
        if not 0 <= r.time_s < n_samples/sample_rate: flags.append('down_outside_audio')
        start = r.time_s-PRE_SECONDS
        end = max(r.time_s+MIN_POST_DOWN_SECONDS,
                  r.up_s+POST_RELEASE_SECONDS if pd.notna(r.up_s) else r.time_s+MIN_POST_DOWN_SECONDS)
        if start < 0 or end > n_samples/sample_rate: flags.append('truncated_at_file_edge')
        starts.append(max(0,min(n_samples,int(np.floor(start*sample_rate)))))
        ends.append(max(0,min(n_samples,int(np.ceil(end*sample_rate)))))
        statuses.append(flags)
    table['start_sample']=starts; table['end_sample']=ends
    # Flag any collisions, including nested windows. Keep proposed boundaries visible for diagnosis.
    active=[]
    for i,r in table.iterrows():
        active=[j for j in active if table.at[j,'end_sample'] > r.start_sample]
        for j in active:
            if 'overlap_other_clip' not in statuses[j]: statuses[j].append('overlap_other_clip')
            if 'overlap_other_clip' not in statuses[i]: statuses[i].append('overlap_other_clip')
        active.append(i)
    table['start_s']=table.start_sample/sample_rate
    table['end_s']=table.end_sample/sample_rate
    table['clip_seconds']=table.end_s-table.start_s
    table['status']=['ok' if not f else ';'.join(f) for f in statuses]
    table['eligible']=table.status.eq('ok') & ((table.end_sample-table.start_sample)>=16)
    table.attrs.update(ignored_repeated_downs=ignored_repeats, unmatched_key_ups=unmatched_ups)
    return table

if events.empty:
    raise FileNotFoundError(f'No key events loaded from {csv_path}. Keep WAV and CSV together.')
labeled = make_clip_table(events,fs,len(x))
display(labeled)
display(labeled.groupby('key').agg(presses=('key','size'),eligible_clips=('eligible','sum')))
print('Ignored repeated key-down events:', labeled.attrs['ignored_repeated_downs'])
print('Unmatched key-up events:', labeled.attrs['unmatched_key_ups'])
print('Eligible = timing checks passed, not guaranteed acoustic alignment or recording quality.')

## Select a labeled keystroke automatically
Change only `EVENT_INDEX` to inspect another press. The key name, key-down and key-up times come from the CSV. Green shading shows the actual sample range used by the FFT and spectrogram. Red shading indicates a timing flag; those clips are excluded from optional export.

Inspect several presses near the start and end of each session: the attack should fall inside the shade, the release should be included, and the neighboring press should be outside it. Unexpected audio offset or drift requires review, not relabeling every key.

In [ ]:
EVENT_INDEX = 0
if not 0 <= EVENT_INDEX < len(labeled):
    raise IndexError(f'Choose EVENT_INDEX from 0 to {len(labeled)-1}.')
event = labeled.iloc[EVENT_INDEX]
center = float(event.time_s)
clip_label = str(event.key)
i0, i1 = int(event.start_sample), int(event.end_sample)
clip = x[i0:i1]
if len(clip)<16:
    raise ValueError('Invalid or empty range; inspect event status and synchronization.')
print(f'Key: {clip_label} | down={center:.6f}s | up={event.up_s:.6f}s')
print(f'Audio analyzed: [{i0/fs:.6f}, {i1/fs:.6f}) s; {len(clip)} samples; status={event.status}')
view0=max(0,i0-int(0.15*fs)); view1=min(len(x),i1+int(0.15*fs))
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(np.arange(view0,view1)/fs,x[view0:view1],lw=0.8)
ax.axvspan(i0/fs,i1/fs,alpha=0.18,color='green' if event.eligible else 'red',label=f'Analyzed clip: {clip_label}')
ax.axvline(center,color='r',ls='--',label='CSV key-down')
if pd.notna(event.up_s): ax.axvline(event.up_s,color='purple',ls=':',label='CSV key-up')
ax.axvline(i0/fs,color='green',lw=1); ax.axvline(i1/fs,color='green',lw=1)
ax.set(xlabel='Time from WAV start (s)',ylabel='Amplitude (FS)',title=f'Automatically labeled keystroke: {clip_label}')
ax.legend(); plt.tight_layout(); plt.show()
display(Audio(clip,rate=fs,normalize=False))

## FFT and power spectral density
The FFT uses a Hann window and removes the DC mean. The single-sided amplitude is corrected for the window gain and expressed relative to digital full-scale amplitude (dBFS), not calibrated sound pressure. Welch PSD describes power per Hz. Zero padding is intentionally omitted: frequency-bin spacing is `fs / N`; a longer real clip improves resolution but may include other sounds.

In [ ]:
def amplitude_spectrum(y, sample_rate):
    y = np.asarray(y, dtype=float)
    if len(y) < 4:
        raise ValueError('Need at least four audio samples.')
    window = signal.windows.hann(len(y), sym=False)
    amp = np.abs(np.fft.rfft((y-y.mean())*window))/window.sum()
    if len(y) % 2 == 0:
        amp[1:-1] *= 2
    else:
        amp[1:] *= 2
    return np.fft.rfftfreq(len(y), 1/sample_rate), amp

f, amplitude = amplitude_spectrum(clip, fs)
fw, psd = signal.welch(clip-clip.mean(), fs=fs, window='hann',
                       nperseg=min(1024,len(clip)), scaling='density')
MAX_FREQ_HZ = min(12000, fs/2)  # raise to fs/2 to inspect the complete recorded band
fig, axes = plt.subplots(1,2,figsize=(14,4))
axes[0].plot(f,20*np.log10(np.maximum(amplitude,1e-12)))
axes[0].set(xlim=(0,MAX_FREQ_HZ), xlabel='Frequency (Hz)',
            ylabel='Amplitude (dBFS)',title='Single-sided FFT amplitude')
axes[1].plot(fw,10*np.log10(np.maximum(psd,1e-24)))
axes[1].set(xlim=(0,MAX_FREQ_HZ),xlabel='Frequency (Hz)',
            ylabel='PSD (dB re 1 FS²/Hz)',title='Welch power spectral density')
plt.tight_layout(); plt.show()
print(f'Clip length: {len(clip)/fs:.4f} s; FFT bin spacing: {fs/len(clip):.2f} Hz')

## Spectrogram: when each frequency occurs
This is a linear-frequency STFT spectrogram. It helps distinguish press/release transients and background noise. It is not a mel-spectrogram. Color values are relative to the largest STFT magnitude in this clip, so do not compare absolute loudness across separate plots using this scale.

In [ ]:
nperseg = min(1024,len(clip))
f_stft, t_stft, Z = signal.stft(clip-clip.mean(), fs=fs, window='hann',
                               nperseg=nperseg, noverlap=3*nperseg//4,
                               boundary=None, padded=False)
mag = np.abs(Z)
relative_db = 20*np.log10(np.maximum(mag,1e-12)/max(float(mag.max()),1e-12))
plt.figure(figsize=(12,5))
plt.pcolormesh(t_stft+i0/fs, f_stft, relative_db, shading='auto',
               cmap='magma', vmin=-80, vmax=0)
plt.ylim(0,MAX_FREQ_HZ)
plt.xlabel('Time from WAV start (s)'); plt.ylabel('Frequency (Hz)')
plt.title(f'STFT spectrogram: {clip_label}')
plt.colorbar(label='dB relative to clip maximum'); plt.tight_layout(); plt.show()

## Compare spectra of automatically labeled keys
Uses the same paired-event clip boundaries shown above. One eligible example per requested key in the selected session; a one-key session will show one curve. Clip durations can vary with hold time, so FFT bin spacing may differ. Use multiple examples when assessing consistency.

In [ ]:
COMPARE_KEYS = ['a','s','d','f','g','h','j','k','l','space']
fig,ax=plt.subplots(figsize=(12,4)); count=0
for key in COMPARE_KEYS:
    matches=labeled[labeled.key.str.lower().eq(key.lower()) & labeled.eligible]
    if matches.empty: continue
    r=matches.iloc[0]
    freq,amp=amplitude_spectrum(x[int(r.start_sample):int(r.end_sample)],fs)
    ax.plot(freq,20*np.log10(np.maximum(amp,1e-12)),label=str(r.key),alpha=0.8)
    count+=1
ax.set(xlim=(0,MAX_FREQ_HZ),xlabel='Frequency (Hz)',ylabel='Amplitude (dBFS)',
       title='Automatically labeled keystroke spectra')
if count: ax.legend()
else: print('No eligible matches. Available keys:',labeled.key.unique())
plt.tight_layout(); plt.show()

## Process every session without manual labeling
The next cell scans all discovered WAVs and their accompanying CSVs. Labels and boundaries are generated automatically. It prints all errors rather than silently skipping malformed sessions. Keep a session's WAV, CSV and JSON together. Multiple WAVs sharing one CSV require review, so those folders are rejected to prevent duplicate labeling.

Set `EXPORT_CLIPS = True` only after spot-checking alignment. Export preserves all original audio channels, creates a manifest with source/session/key/timing/status, and exports only clips that pass structural timing checks. Spectra and silence/noise still require inspection. The first run does not export. For repeated exports use a new output folder; existing clips are never overwritten.

In [ ]:
EXPORT_CLIPS = False
OUTPUT_DIR = ROOT.parent / 'KeyboardAudioProcessed'

def scan_sessions(wav_paths):
    tables=[]; errors=[]
    parent_counts={p.parent:sum(q.parent==p.parent for q in wav_paths) for p in wav_paths}
    for wav in wav_paths:
        session_id=wav.parent.relative_to(ROOT).as_posix()
        try:
            if parent_counts[wav.parent] != 1:
                raise ValueError('Multiple WAVs in one session folder; WAV/CSV association is ambiguous.')
            info=sf.info(str(wav))
            event_file=wav.parent/'key_events.csv'
            if not event_file.exists(): raise FileNotFoundError('Missing key_events.csv')
            metadata_file=wav.parent/'session.json'
            if metadata_file.exists():
                metadata=json.loads(metadata_file.read_text(encoding='utf-8-sig'))
                recorded_fs=metadata.get('sample_rate_hz')
                if recorded_fs is not None and float(recorded_fs)!=info.samplerate:
                    raise ValueError('session.json sample rate does not match WAV.')
            table=make_clip_table(pd.read_csv(event_file),info.samplerate,info.frames)
            table.insert(0,'session_id',session_id)
            table.insert(1,'wav_path',str(wav))
            table['sample_rate']=info.samplerate
            table['channels']=info.channels
            tables.append(table)
        except Exception as exc:
            errors.append(dict(session_id=session_id,wav_path=str(wav),error=str(exc)))
    return (pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()),pd.DataFrame(errors)

manifest,session_errors=scan_sessions(wavs)
if not session_errors.empty:
    print('Sessions needing attention:'); display(session_errors)
if manifest.empty:
    print('No session could be processed. Check errors above.')
else:
    display(manifest.groupby(['session_id','key']).agg(presses=('key','size'),eligible=('eligible','sum')))
    print('Status counts:'); display(manifest.status.value_counts().to_frame('count'))
    if EXPORT_CLIPS:
        if OUTPUT_DIR.exists():
            raise FileExistsError(f'{OUTPUT_DIR} exists. Choose a new OUTPUT_DIR to avoid overwriting files.')
        OUTPUT_DIR.mkdir(parents=True)
        manifest['export_path']=''
        # Session-relative folders retain provenance; numeric filenames avoid Windows key-name restrictions.
        for wav_name,group in manifest.groupby('wav_path',sort=False):
            with sf.SoundFile(wav_name) as source:
                for idx,r in group.iterrows():
                    if not r.eligible: continue
                    dest=OUTPUT_DIR/str(r.session_id)/f'clip_{idx:06d}.wav'
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    source.seek(int(r.start_sample))
                    samples=source.read(int(r.end_sample-r.start_sample),dtype='float64',always_2d=True)
                    if len(samples)!=int(r.end_sample-r.start_sample):
                        raise ValueError(f'Incomplete source audio: {wav_name}')
                    sf.write(str(dest),samples,int(r.sample_rate),subtype='FLOAT')
                    manifest.at[idx,'export_path']=str(dest)
        manifest.to_csv(OUTPUT_DIR/'clip_manifest.csv',index=False)
        session_errors.to_csv(OUTPUT_DIR/'session_errors.csv',index=False)
        print('Exported to:',OUTPUT_DIR)
    else:
        print('Preview only. After alignment checks, set EXPORT_CLIPS=True to write clips and manifest.')

## Training next
Split by whole session before fitting or augmenting. The manifest carries key labels and session IDs. If each key was recorded in a separate folder, a balanced train/test split requires several recording rounds containing all keys; do not hold out an entire key class accidentally. Variable-length clips can be padded or transformed to a consistent feature size during training. Keep original WAVs and the manifest for reproducibility.